In [2]:
"""Generate the 4 new FC builder notebooks + PATCH file.

Preserves the exact conventions of the existing build_product_limits.ipynb:
- Cell 0: preamble, imports, paths, source load, SHA
- Cell 1: helpers
- Cell 2: harvest + assemble all_rows
- Cell 3: type classify + polars build + structural validation + frozen expectations
- Cell 4: write parquet + upsert master workbook sheet
- Cell 5: append build_log.csv entry
"""
import json
from pathlib import Path

# Where the generated notebooks + patch file will be written.
# Change this to any folder you like — the script will create it if missing.
OUT_DIR = Path(r"\\omsmds000\eb3\Group_Assurance\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\fc_builders")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def nb(cells):
    return {
        "cells": [
            {"cell_type": "code", "execution_count": None, "metadata": {},
             "outputs": [], "source": src.splitlines(keepends=True)}
            for src in cells
        ],
        "metadata": {
            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
            "language_info": {"name": "python", "version": "3.11"},
        },
        "nbformat": 4, "nbformat_minor": 5,
    }


def write_nb(name, cells):
    path = OUT_DIR / name
    path.write_text(json.dumps(nb(cells), indent=1))
    print(f"  ✓ {name}")


HELPERS = '''\
def find_all_cells_any_col(ws, label: str, max_row: int = 1200, max_col: int = 30) -> list[tuple[int, int]]:
    """Find ALL cells matching `label` exactly, anywhere on the sheet."""
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col))
    return hits


def find_in_range(ws, label: str, row_start: int, row_end: int, col: int | None = None) -> tuple[int, int] | None:
    """Find first cell matching `label` within given row range (and optional col)."""
    hits = find_all_cells_any_col(ws, label)
    for r, c in hits:
        if row_start <= r < row_end and (col is None or c == col):
            return (r, c)
    return None


def get_col_letter(col: int) -> str:
    return openpyxl.utils.get_column_letter(col)
'''

# ============================================================================
# 1. build_fc_rates.ipynb   →   basis snapshot
# ============================================================================
fc_rates_cells = [
    '''\
"""
Build basis_master sheet: fc_rates
-----
Family Cover children rate schedule (2 age bands): 0-13 basic, 14+ basic + aids.
Source: Basis and Assumptions.xlsx > FC Basis - New > C3:E5.

FC row-4 formula wiring (V1.7.43):
  AW (kids 0-13 basic)   = 'FC Basis - New'!$D$4 * n_children_0_13
  AX (kids 14+ basic)    = 'FC Basis - New'!$D$5 * n_children_14_plus
  AY (kids 14+ aids)     = 'FC Basis - New'!$E$5 * n_children_14_plus

Author: Mohamed Hussein Gadiwala
Basis version: v2026_07 (basis master)
"""

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib
import csv
import openpyxl
import polars as pl

BASE = Path(r"J:\\Product & Client Solutions\\Toolbox\\Calculators"
            r"\\Business Tools\\Tender Summary\\GAP Pricing Engine")

SOURCE_XLSX  = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "FC Basis - New"

BASIS_VERSION = "v2026_07"
SNAPSHOT_DIR = BASE / "basis_snapshots" / f"basis_{BASIS_VERSION}"
MASTER_XLSX  = BASE / "basis_master" / f"basis_master_{BASIS_VERSION}.xlsx"
BUILD_LOG    = SNAPSHOT_DIR / "build_log.csv"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

SHEET_NAME   = "fc_rates"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")
print(f"Source:   {SOURCE_XLSX.name} > {SOURCE_SHEET}")
print(f"Target:   {PARQUET_PATH}")
print(f"Loaded:   {ws.max_row} rows × {ws.max_column} cols")
print(f"Built at: {BUILT_AT}")
''',
    HELPERS,
    '''\
# Anchor: FC Basis - New!C3:E5 has header "START_AGE | BASIC_RATE | AIDS_RATE"
expected_headers = ["START_AGE", "BASIC_RATE", "AIDS_RATE"]
actual_headers = [ws.cell(row=3, column=c).value for c in (3, 4, 5)]
assert actual_headers == expected_headers, (
    f"FC Basis - New!C3:E3 header drift: got {actual_headers}, expected {expected_headers}"
)

all_rows: list[dict] = []
for excel_row in (4, 5):
    age_from   = ws.cell(row=excel_row, column=3).value
    basic_rate = ws.cell(row=excel_row, column=4).value
    aids_rate  = ws.cell(row=excel_row, column=5).value

    assert isinstance(age_from, (int, float))
    assert isinstance(basic_rate, (int, float))
    assert isinstance(aids_rate, (int, float))

    all_rows.append({
        "product":       "fc",
        "age_from":      int(age_from),
        "basic_rate":    float(basic_rate),
        "aids_rate":     float(aids_rate),
        "source_label":  f"FC Basis - New!D{excel_row}, E{excel_row}",
        "source_row":    excel_row,
        "notes": ("Children basic + aids rates per R1000 cover. "
                  "Applied by FC pricing via n_children_0_13 and n_children_14_plus."),
    })

print(f"Total rows harvested: {len(all_rows)}")
for r in all_rows:
    print(f"  age_from={r['age_from']:>3d}  basic={r['basic_rate']:.6f}  "
          f"aids={r['aids_rate']:.6f}  src=row{r['source_row']}")
''',
    '''\
df = pl.DataFrame(all_rows).select([
    "product", "age_from", "basic_rate", "aids_rate",
    "source_label", "source_row", "notes",
])

key_cols = ["product", "age_from"]
dupes = df.group_by(key_cols).len().filter(pl.col("len") > 1)
assert dupes.height == 0, f"Duplicate keys:\\n{dupes}"

expected_products = {"fc"}
actual_products = set(df["product"].unique().to_list())
assert actual_products == expected_products

def get_rate(age_from, col):
    row = df.filter(pl.col("age_from") == age_from)
    assert row.height == 1
    return float(row[col][0])

assert abs(get_rate(0, "basic_rate")  - 1.11913253117143) < 1e-9
assert abs(get_rate(0, "aids_rate")   - 0.0) < 1e-9
assert abs(get_rate(14, "basic_rate") - 0.651755343823977) < 1e-9
assert abs(get_rate(14, "aids_rate")  - 0.378189613982071) < 1e-9

print(f"✓ {df.height} rows validated")
print(f"✓ 2 age bands present (0-13, 14+)")
print(f"✓ 4 frozen numeric checks passed")
print(f"\\nFinal DataFrame:")
print(df)
''',
    '''\
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name}")
print(f"  Size:    {PARQUET_SIZE_KB:.2f} KB")
print(f"  SHA-256: {PARQUET_SHA[:16]}...")

if MASTER_XLSX.exists():
    master_wb = openpyxl.load_workbook(MASTER_XLSX)
else:
    master_wb = openpyxl.Workbook()
    if "Sheet" in master_wb.sheetnames:
        del master_wb["Sheet"]

if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated: {MASTER_XLSX.name}")
''',
    '''\
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "basis_version", "notes",
]

log_row = {
    "built_at": BUILT_AT, "built_by": BUILT_BY, "sheet": SHEET_NAME,
    "rows": df.height, "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}", "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name, "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA, "basis_version": BASIS_VERSION,
    "notes": ("FC children rate schedule. 2 age bands (0-13, 14+). "
              "Anchored on FC Basis - New!C3:E5. Per R1000 cover. "
              "Consumer: FC row-4 formulas AW/AX/AY."),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated ({BUILD_LOG.name})")
print(f"\\n{'='*60}\\nBUILD COMPLETE: {SHEET_NAME}\\n{'='*60}")
''',
]
write_nb("build_fc_rates.ipynb", fc_rates_cells)


# ============================================================================
# 2. build_fc_demographics.ipynb   →   basis snapshot
# ============================================================================
fc_demo_cells = [
    '''\
"""
Build basis_master sheet: fc_demographics
-----
Family Cover demographic assumptions by age: pct_married, n_children_0_13, n_children_14_plus.
Source: Basis and Assumptions.xlsx > FC Basis - New > G3:J103.

Header row: ANB | MARRIAGE_PERC | AGED_0_TO_13 | AGED_14_PLUS
Age range: 16..115 (100 rows).

Normalization: MARRIAGE_PERC source stores 86.0 meaning 86%. Normalised to decimal here.

Author: Mohamed Hussein Gadiwala
Basis version: v2026_07 (basis master)
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import csv
import openpyxl
import polars as pl

BASE = Path(r"J:\\Product & Client Solutions\\Toolbox\\Calculators"
            r"\\Business Tools\\Tender Summary\\GAP Pricing Engine")

SOURCE_XLSX  = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "FC Basis - New"

BASIS_VERSION = "v2026_07"
SNAPSHOT_DIR = BASE / "basis_snapshots" / f"basis_{BASIS_VERSION}"
MASTER_XLSX  = BASE / "basis_master" / f"basis_master_{BASIS_VERSION}.xlsx"
BUILD_LOG    = SNAPSHOT_DIR / "build_log.csv"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

SHEET_NAME   = "fc_demographics"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")
print(f"Source:   {SOURCE_XLSX.name} > {SOURCE_SHEET}")
print(f"Target:   {PARQUET_PATH}")
print(f"Built at: {BUILT_AT}")
''',
    HELPERS,
    '''\
expected_headers = ["ANB", "MARRIAGE_PERC", "AGED_0_TO_13", "AGED_14_PLUS"]
actual_headers = [ws.cell(row=3, column=c).value for c in (7, 8, 9, 10)]
assert actual_headers == expected_headers, (
    f"Header drift: got {actual_headers}, expected {expected_headers}"
)

DATA_ROW_START = 4
DATA_ROW_END   = 104
EXPECTED_AGES  = list(range(16, 116))

all_rows: list[dict] = []
for excel_row in range(DATA_ROW_START, DATA_ROW_END):
    age_raw          = ws.cell(row=excel_row, column=7).value
    marriage_raw     = ws.cell(row=excel_row, column=8).value
    kids_0_13_raw    = ws.cell(row=excel_row, column=9).value
    kids_14_plus_raw = ws.cell(row=excel_row, column=10).value

    if age_raw is None:
        raise RuntimeError(f"Null age at row {excel_row}")

    all_rows.append({
        "product":            "fc",
        "age":                int(age_raw),
        "pct_married":        float(marriage_raw) / 100.0 if marriage_raw is not None else 0.0,
        "n_children_0_13":    float(kids_0_13_raw)    if kids_0_13_raw    is not None else 0.0,
        "n_children_14_plus": float(kids_14_plus_raw) if kids_14_plus_raw is not None else 0.0,
    })

print(f"Total rows harvested: {len(all_rows)}")
print("First 3:", all_rows[:3])
print("Last 3:",  all_rows[-3:])
''',
    '''\
df = pl.DataFrame(all_rows).select([
    "product", "age", "pct_married", "n_children_0_13", "n_children_14_plus",
])

dupes = df.group_by(["product", "age"]).len().filter(pl.col("len") > 1)
assert dupes.height == 0

assert set(df["product"].unique().to_list()) == {"fc"}

ages = sorted(df["age"].to_list())
assert ages == EXPECTED_AGES, f"Age drift: {ages[0]}..{ages[-1]}"

assert df["pct_married"].max() <= 1.0
assert df["pct_married"].min() >= 0.0
assert df["n_children_0_13"].min() >= 0.0
assert df["n_children_14_plus"].min() >= 0.0

def get_row(age):
    row = df.filter(pl.col("age") == age)
    return row.row(0, named=True)

r = get_row(16)
assert r["pct_married"] == 0.0 and r["n_children_0_13"] == 1.0

r = get_row(40)
assert abs(r["pct_married"] - 0.86) < 1e-9
assert r["n_children_0_13"] == 3.0 and r["n_children_14_plus"] == 1.0

r = get_row(65)
assert abs(r["pct_married"] - 0.85) < 1e-9
assert r["n_children_0_13"] == 0.0

r = get_row(115)
assert abs(r["pct_married"] - 0.49) < 1e-9

print(f"✓ {df.height} rows validated")
print(f"✓ ages contiguous 16..115")
print(f"✓ pct_married normalised (max={df['pct_married'].max():.2f})")
print(f"✓ frozen checks passed")
print(df.head(3))
print(df.tail(3))
''',
    '''\
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name} ({PARQUET_SIZE_KB:.2f} KB)")

if MASTER_XLSX.exists():
    master_wb = openpyxl.load_workbook(MASTER_XLSX)
else:
    master_wb = openpyxl.Workbook()
    if "Sheet" in master_wb.sheetnames:
        del master_wb["Sheet"]

if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated")
''',
    '''\
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "basis_version", "notes",
]

log_row = {
    "built_at": BUILT_AT, "built_by": BUILT_BY, "sheet": SHEET_NAME,
    "rows": df.height, "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}", "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name, "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA, "basis_version": BASIS_VERSION,
    "notes": ("FC demographics by age (16..115, 100 rows). pct_married "
              "normalised to decimal at build time. Consumer: FC row-4 AB/AC/AD."),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\\n{'='*60}\\nBUILD COMPLETE: {SHEET_NAME}\\n{'='*60}")
''',
]
write_nb("build_fc_demographics.ipynb", fc_demo_cells)


# ============================================================================
# 3. build_fc_family_composition.ipynb   →   basis snapshot
# ============================================================================
fc_fam_cells = [
    '''\
"""
Build basis_master sheet: fc_family_composition
-----
FC parents/in-laws by age band (step function).
Source: Basis and Assumptions.xlsx > FC Basis > BG3:BJ13.

Note: col BH ('Premium per Member') is LEGACY — intentionally dropped.

FC row-4 formula wiring (V1.7.43):
  AE (# Parents) = VLOOKUP(G4, FC_Parents, 3, TRUE)   -> BI column
  AF (# In-Laws) = VLOOKUP(G4, FC_Parents, 4, TRUE)   -> BJ column
Named range FC_Parents = 'FC Basis'!$BG$4:$BJ$13.

Author: Mohamed Hussein Gadiwala
Basis version: v2026_07 (basis master)
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import csv
import openpyxl
import polars as pl

BASE = Path(r"J:\\Product & Client Solutions\\Toolbox\\Calculators"
            r"\\Business Tools\\Tender Summary\\GAP Pricing Engine")

SOURCE_XLSX  = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "FC Basis"

BASIS_VERSION = "v2026_07"
SNAPSHOT_DIR = BASE / "basis_snapshots" / f"basis_{BASIS_VERSION}"
MASTER_XLSX  = BASE / "basis_master" / f"basis_master_{BASIS_VERSION}.xlsx"
BUILD_LOG    = SNAPSHOT_DIR / "build_log.csv"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

SHEET_NAME   = "fc_family_composition"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")
print(f"Source:   {SOURCE_XLSX.name} > {SOURCE_SHEET}")
print(f"Target:   {PARQUET_PATH}")
print(f"Built at: {BUILT_AT}")
''',
    HELPERS,
    '''\
# BG=59, BH=60 (legacy, dropped), BI=61, BJ=62
COL_AGE, COL_PARENTS, COL_INLAWS = 59, 61, 62

header_age     = ws.cell(row=3, column=COL_AGE).value
header_parents = ws.cell(row=3, column=COL_PARENTS).value
header_inlaws  = ws.cell(row=3, column=COL_INLAWS).value

assert isinstance(header_age, str) and "age" in header_age.lower()
assert isinstance(header_parents, str) and "parent" in header_parents.lower()
assert isinstance(header_inlaws, str) and "law" in header_inlaws.lower()

DATA_ROW_START = 4
DATA_ROW_END   = 14
EXPECTED_AGES  = [0, 25, 30, 35, 40, 45, 50, 55, 60, 65]

all_rows: list[dict] = []
for excel_row in range(DATA_ROW_START, DATA_ROW_END):
    age       = ws.cell(row=excel_row, column=COL_AGE).value
    n_parents = ws.cell(row=excel_row, column=COL_PARENTS).value
    n_in_laws = ws.cell(row=excel_row, column=COL_INLAWS).value

    assert isinstance(age, (int, float))
    assert isinstance(n_parents, (int, float))
    assert isinstance(n_in_laws, (int, float))

    all_rows.append({
        "product":   "fc",
        "age_from":  int(age),
        "n_parents": float(n_parents),
        "n_in_laws": float(n_in_laws),
    })

print(f"Total rows harvested: {len(all_rows)}")
print("(source col BH 'Premium per Member' dropped — legacy)")
for r in all_rows:
    print(f"  age_from={r['age_from']:>2d}  n_parents={r['n_parents']}  n_in_laws={r['n_in_laws']}")
''',
    '''\
df = pl.DataFrame(all_rows).select([
    "product", "age_from", "n_parents", "n_in_laws",
])

dupes = df.group_by(["product", "age_from"]).len().filter(pl.col("len") > 1)
assert dupes.height == 0

assert set(df["product"].unique().to_list()) == {"fc"}

ages = sorted(df["age_from"].to_list())
assert ages == sorted(EXPECTED_AGES)

assert df["n_parents"].min() >= 0 and df["n_parents"].max() <= 5
assert df["n_in_laws"].min() >= 0 and df["n_in_laws"].max() <= 5

def get_row(age_from):
    return df.filter(pl.col("age_from") == age_from).row(0, named=True)

r = get_row(0);  assert r["n_parents"] == 2.0 and r["n_in_laws"] == 0.0
r = get_row(25); assert r["n_parents"] == 2.0 and r["n_in_laws"] == 2.0
r = get_row(40); assert r["n_parents"] == 1.5 and r["n_in_laws"] == 1.5
r = get_row(65); assert r["n_parents"] == 0.5 and r["n_in_laws"] == 0.5

parents_by_age = df.sort("age_from")["n_parents"].to_list()
for i in range(1, len(parents_by_age)):
    assert parents_by_age[i] <= parents_by_age[i-1]

print(f"✓ {df.height} rows validated")
print(f"✓ age bands: {ages}")
print(f"✓ 8 frozen numeric checks passed")
print(f"✓ n_parents monotonic non-increasing")
print(df)
''',
    '''\
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name} ({PARQUET_SIZE_KB:.2f} KB)")

if MASTER_XLSX.exists():
    master_wb = openpyxl.load_workbook(MASTER_XLSX)
else:
    master_wb = openpyxl.Workbook()
    if "Sheet" in master_wb.sheetnames:
        del master_wb["Sheet"]

if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated")
''',
    '''\
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "basis_version", "notes",
]

log_row = {
    "built_at": BUILT_AT, "built_by": BUILT_BY, "sheet": SHEET_NAME,
    "rows": df.height, "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}", "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name, "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA, "basis_version": BASIS_VERSION,
    "notes": ("FC parents/in-laws by age (10 bands, 0..65 step-5). "
              "Named range FC_Parents. Col BH 'Premium per Member' dropped (legacy). "
              "Consumer: FC row-4 AE/AF."),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\\n{'='*60}\\nBUILD COMPLETE: {SHEET_NAME}\\n{'='*60}")
''',
]
write_nb("build_fc_family_composition.ipynb", fc_fam_cells)


# ============================================================================
# 4. build_fc_db_loading.ipynb   →   assumption snapshot
# ============================================================================
fc_db_cells = [
    '''\
"""
Build assumption_master sheet: fc_db_loading
-----
FC DB loading — multiplier for members with Status='DB'. 5-row step function.
Source: Basis and Assumptions.xlsx > Assumptions > E418:F422.
Anchor label: 'Months of Disability' at Assumptions!E417.

FC row-4 formula wiring (V1.7.43):
  BA (DB Loading) = INDEX(Assumptions!$E$418:$F$422,
                          MIN(5, FLOOR((Comm_Date - F4)/365.25, 1) + 1), 2)
Positional index (MIN cap 5) → years_disabled 0..4.

Author: Mohamed Hussein Gadiwala
Assumption version: A2026_07 (assumption master)
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import csv
import openpyxl
import polars as pl

BASE = Path(r"J:\\Product & Client Solutions\\Toolbox\\Calculators"
            r"\\Business Tools\\Tender Summary\\GAP Pricing Engine")

SOURCE_XLSX  = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX  = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG    = SNAPSHOT_DIR / "build_log.csv"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

SHEET_NAME   = "fc_db_loading"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")
print(f"Source:   {SOURCE_XLSX.name} > {SOURCE_SHEET}")
print(f"Built at: {BUILT_AT}")
''',
    HELPERS,
    '''\
ANCHOR_LABEL = "Months of Disability"
hit = find_in_range(ws, ANCHOR_LABEL, 415, 425, col=5)
assert hit is not None, f"Anchor {ANCHOR_LABEL!r} not found"
anchor_row, anchor_col = hit
print(f"Anchor at Assumptions!{get_col_letter(anchor_col)}{anchor_row}")

DATA_ROW_START = anchor_row + 1
DATA_ROW_END   = anchor_row + 6

EXPECTED_LABELS = ["0 to 12", "12 to 24", "24 to 36", "36 to 48", "> 48"]

all_rows: list[dict] = []
for i, excel_row in enumerate(range(DATA_ROW_START, DATA_ROW_END)):
    source_label   = ws.cell(row=excel_row, column=anchor_col).value
    loading_factor = ws.cell(row=excel_row, column=anchor_col + 1).value

    assert isinstance(source_label, str)
    assert isinstance(loading_factor, (int, float))
    label_norm = source_label.strip()
    assert label_norm == EXPECTED_LABELS[i], (
        f"Row {excel_row}: label drift, got {label_norm!r}, expected {EXPECTED_LABELS[i]!r}"
    )

    all_rows.append({
        "product":        "fc",
        "years_disabled": i,
        "loading_factor": float(loading_factor),
        "source_label":   label_norm,
        "source_row":     excel_row,
        "notes":          "FC DB loading applied when Status='DB'. Positional; row 4 = 4+ years.",
    })

print(f"Total rows: {len(all_rows)}")
for r in all_rows:
    print(f"  years_disabled={r['years_disabled']}  loading={r['loading_factor']:.2f}  "
          f"({r['source_label']!r})")
''',
    '''\
df = pl.DataFrame(all_rows).select([
    "product", "years_disabled", "loading_factor",
    "source_label", "source_row", "notes",
])

dupes = df.group_by(["product", "years_disabled"]).len().filter(pl.col("len") > 1)
assert dupes.height == 0
assert set(df["product"].unique().to_list()) == {"fc"}

years = sorted(df["years_disabled"].to_list())
assert years == [0, 1, 2, 3, 4]

factors = df.sort("years_disabled")["loading_factor"].to_list()
for i in range(1, len(factors)):
    assert factors[i] <= factors[i-1]

def get_loading(years_disabled):
    return float(df.filter(pl.col("years_disabled") == years_disabled)["loading_factor"][0])

assert abs(get_loading(0) - 14.5) < 1e-6
assert abs(get_loading(1) -  9.0) < 1e-6
assert abs(get_loading(2) -  8.0) < 1e-6
assert abs(get_loading(3) -  3.5) < 1e-6
assert abs(get_loading(4) -  1.5) < 1e-6

print(f"✓ {df.height} rows validated")
print(f"✓ monotonically non-increasing: {factors}")
print(f"✓ 5 frozen checks passed")
print(df)
''',
    '''\
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name} ({PARQUET_SIZE_KB:.2f} KB)")

if MASTER_XLSX.exists():
    master_wb = openpyxl.load_workbook(MASTER_XLSX)
else:
    master_wb = openpyxl.Workbook()
    if "Sheet" in master_wb.sheetnames:
        del master_wb["Sheet"]

if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated")
''',
    '''\
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "assumption_version", "notes",
]

log_row = {
    "built_at": BUILT_AT, "built_by": BUILT_BY, "sheet": SHEET_NAME,
    "rows": df.height, "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}", "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name, "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA, "assumption_version": ASSUMPTION_VERSION,
    "notes": ("FC DB loading (5-row step function). Positional years_disabled 0..4. "
              "Anchor: Assumptions!E417 'Months of Disability'. Consumer: FC row-4 BA."),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\\n{'='*60}\\nBUILD COMPLETE: {SHEET_NAME}\\n{'='*60}")
''',
]
write_nb("build_fc_db_loading.ipynb", fc_db_cells)



  ✓ build_fc_rates.ipynb
  ✓ build_fc_demographics.ipynb
  ✓ build_fc_family_composition.ipynb
  ✓ build_fc_db_loading.ipynb
